<a href="https://colab.research.google.com/github/lilianeuwase2/math_forML-Formative3/blob/main/ProbabilityDistributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import csv
import numpy as np

mothers = []
children = []
seen_families = set()
with open('GaltonFamilies.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        children.append(float(row['childHeight']))
        if row["family"] not in seen_families:
          seen_families.add(row["family"])
          mothers.append(float(row['mother']))
children = np.array(children)
mothers = np.array(mothers)

heights = np.concatenate([mothers, children])
print(len(mothers), len(children), len(heights))

205 934 1139


Gaussian Function


In [12]:
def gaussian(heights, mu, var):
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-(heights- mu) ** 2 / (2 * var))

Log-Likelihood

In [13]:
def log_likelihood(heights, mu1, mu2, var1, var2, pi1, pi2):
    total = pi1 * gaussian(heights, mu1, var1) + pi2 * gaussian(heights, mu2, var2)
    return np.sum(np.log(total))

E-step separate

In [14]:
def e_step(heights, mu1, mu2, var1, var2, pi1, pi2):
    score1 = pi1 * gaussian(heights, mu1, var1)
    score2 = pi2 * gaussian(heights, mu2, var2)
    total = score1 + score2
    return score1 / total, score2 / total

M-step separate

In [15]:
def m_step(heights, prob1, prob2):
    N = len(heights)
    pi1 = prob1.sum() / N
    pi2 = prob2.sum() / N
    mu1 = np.sum(prob1 * heights) / prob1.sum()
    mu2 = np.sum(prob2 * heights) / prob2.sum()
    var1 = np.sum(prob1 * (heights - mu1) ** 2) / prob1.sum()
    var2 = np.sum(prob2 * (heights - mu2) ** 2) / prob2.sum()
    return mu1, mu2, var1, var2, pi1, pi2

the Loop

In [16]:
def initial_params(heights): #this function helps us to initiate the first guesses once
    mu1 = heights.mean() - heights.std()
    mu2 = heights.mean() + heights.std()
    var1 = heights.var()
    var2 = heights.var()
    pi1 = 0.5
    pi2 = 0.5
    return mu1, mu2, var1, var2, pi1, pi2

In [17]:
def run_em(heights, n_iter):
    params = (initial_params(heights))
    history = []

    for iteration in range(n_iter + 1):
      if iteration > 0:
        prob1, prob2 = e_step(heights, *params)
        params = m_step(heights, prob1, prob2)
      history.append([iteration, *params, log_likelihood(heights, *params)])

      if iteration > 0 and abs(history[-1][-1] - history[-2][-1]) < 1e-6:
        break

    return history, params

Tracking table

In [18]:
history_table, _ = run_em(heights, 2)

print("iter   mu1   mu2   var1    var2    pi1   pi2   log_lik")
for row in history_table:
    print(f"{row[0]:>4} " + " ".join(f"{v:9.4f}" for v in row[1:]))


iter   mu1   mu2   var1    var2    pi1   pi2   log_lik
   0   62.7052   69.7989   12.5802   12.5802    0.5000    0.5000 -3190.3325
   1   64.0693   68.5283    6.5015    8.7695    0.5105    0.4895 -3041.3709
   2   64.0394   68.5861    6.0192    8.8886    0.5133    0.4867 -3039.7574


In [22]:
history, params = run_em(heights, 1000)

mu1, mu2, var1, var2, pi1, pi2 = params
mother_average, child_average = np.mean(mothers), np.mean(children)

if abs(mu1 - child_average) < abs(mu1 - mother_average):
    label_group1, label_group2 = "Child", "Parent"
else:
    label_group1, label_group2 = "Parent", "Child"

print(f"mothers' average {mother_average:.2f}, children's average {child_average:.2f}")
print(f"group 1 mean {mu1:.2f} -> {label_group1}")
print(f"group 2 mean {mu2:.2f} -> {label_group2}")

mothers' average 64.00, children's average 66.75
group 1 mean 64.30 -> Parent
group 2 mean 69.79 -> Child


In [20]:
def predict_height(test_height):
    prob1, prob2 = e_step(np.array([test_height]), *params)
    probabilities = {label_group1: prob1[0], label_group2: prob2[0]}
    print(f"Height {test_height} inches:")
    print(f"  P(Child)  = {probabilities['Child']:.6f}")
    print(f"  P(Parent) = {probabilities['Parent']:.6f}")
    print(f"  Sum       = {probabilities['Child'] + probabilities['Parent']:.6f}")

predict_height(60)
predict_height(66)
predict_height(72)

Height 60 inches:
  P(Child)  = 0.000446
  P(Parent) = 0.999554
  Sum       = 1.000000
Height 66 inches:
  P(Child)  = 0.164667
  P(Parent) = 0.835333
  Sum       = 1.000000
Height 72 inches:
  P(Child)  = 0.983782
  P(Parent) = 0.016218
  Sum       = 1.000000
